**# EE4146 MACHINE LEARNING PROJECT:**

**## RESEARCH QUESTION:**

Does the optimal historical time window for predicting future Fantasy Premier League (FPL) points vary depending on a player's position (Goalkeeper, Defender, Midfielder, Forward)?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os

In [ ]:
def preprocess_season(raw_filename, season_tag):
    """
    raw_filename: name of the CSV in Raw_Data (e.g. 'raw_24_25.csv')
    season_tag: short label AND subfolder name (e.g. '24_25')
    """
    # 1. Load raw data
    file_path = f'/content/drive/MyDrive/EE4146_FPL_Project/Raw_Data/{raw_filename}'
    df = pd.read_csv(file_path)
    print(f'\n===== Processing season {season_tag} from {raw_filename} =====')
    print(f'Total raw rows: {len(df)}')

    # 2. Keep only needed columns
    features_to_keep = [
        'name', 'position', 'team', 'GW',
        'total_points',
        'goals_scored', 'assists', 'minutes',
        'expected_goals', 'expected_assists',
        'fixture', 'was_home', 'clean_sheets'
    ]
    df_clean = df[features_to_keep].copy()

    # >>> NEW: add season column <<<
    df_clean['season'] = season_tag

    # 3. Sort chronologically per player
    df_clean = df_clean.sort_values(by=['name', 'GW']).reset_index(drop=True)

    # 4. Engineer 4-game rolling "form"
    df_clean['engineered_form'] = df_clean.groupby('name')['total_points'].transform(
        lambda x: x.rolling(window=4, min_periods=1).mean()
    )

    # 5. Drop bench rows (0 minutes)
    rows_before = len(df_clean)
    df_clean = df_clean[df_clean['minutes'] > 0].copy()
    rows_after = len(df_clean)
    print(f'Removed {rows_before - rows_after} bench rows.')
    print(f'Remaining active rows: {rows_after}')

    # 6. Create target: next gameweek points
    df_clean['Next_GW_Points'] = df_clean.groupby('name')['total_points'].shift(-1)
    df_clean = df_clean.dropna(subset=['Next_GW_Points']).copy()

    # 7. Split by position
    df_gk  = df_clean[df_clean['position'] == 'GK']
    df_def = df_clean[df_clean['position'] == 'DEF']
    df_mid = df_clean[df_clean['position'] == 'MID']
    df_fwd = df_clean[df_clean['position'] == 'FWD']

    # 8. Save into season-specific subfolder
    base_dir = '/content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/'
    season_dir = base_dir + f'{season_tag}/'

    os.makedirs(season_dir, exist_ok=True)

    df_gk.to_csv(season_dir + f'GK_clean_{season_tag}.csv', index=False)
    df_def.to_csv(season_dir + f'DEF_clean_{season_tag}.csv', index=False)
    df_mid.to_csv(season_dir + f'MID_clean_{season_tag}.csv', index=False)
    df_fwd.to_csv(season_dir + f'FWD_clean_{season_tag}.csv', index=False)

    print(f"Exported {len(df_gk)} GK, {len(df_def)} DEF, {len(df_mid)} MID, {len(df_fwd)} FWD rows for season {season_tag}.")
    print(f"Saved to {season_dir}")

In [ ]:
# 24/25 (already done once, but this will reproduce cleanly)
preprocess_season('raw_24_25.csv', '24_25')

# 23/24
preprocess_season('raw_23_24.csv', '23_24')

# 25/26
preprocess_season('raw_25_26.csv', '25_26')


===== Processing season 24_25 from raw_24_25.csv =====
Total raw rows: 27605
Removed 16039 bench rows.
Remaining active rows: 11566
Exported 726 GK, 3589 DEF, 5481 MID, 1208 FWD rows for season 24_25.
Saved to /content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/24_25/

===== Processing season 23_24 from raw_23_24.csv =====
Total raw rows: 29725
Removed 18341 bench rows.
Remaining active rows: 11384
Exported 735 GK, 3648 DEF, 5058 MID, 1370 FWD rows for season 23_24.
Saved to /content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/23_24/

===== Processing season 25_26 from raw_25_26.csv =====
Total raw rows: 22353
Removed 13541 bench rows.
Remaining active rows: 8812
Exported 553 GK, 2859 DEF, 3878 MID, 1004 FWD rows for season 25_26.
Saved to /content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/25_26/


---------------------- Merging All Seasons -----------------------

In [ ]:
base_dir = '/content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/'
seasons = ['23_24', '24_25', '25_26']

def combine_position(position):
    dfs = []
    for season in seasons:
        path = os.path.join(base_dir, season, f'{position}_clean_{season}.csv')
        df = pd.read_csv(path)
        # safety: if some old file missed the season column, add it
        if 'season' not in df.columns:
            df['season'] = season
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.sort_values(['name', 'season', 'GW']).reset_index(drop=True)
    return combined

df_gk_all  = combine_position('GK')
df_def_all = combine_position('DEF')
df_mid_all = combine_position('MID')
df_fwd_all = combine_position('FWD')

out_dir = os.path.join(base_dir, 'ALL_SEASONS')
os.makedirs(out_dir, exist_ok=True)

df_gk_all.to_csv(os.path.join(out_dir, 'GK_all_seasons.csv'), index=False)
df_def_all.to_csv(os.path.join(out_dir, 'DEF_all_seasons.csv'), index=False)
df_mid_all.to_csv(os.path.join(out_dir, 'MID_all_seasons.csv'), index=False)
df_fwd_all.to_csv(os.path.join(out_dir, 'FWD_all_seasons.csv'), index=False)

print("Saved combined files to:", out_dir)

Saved combined files to: /content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/ALL_SEASONS


In [ ]:
# Merging Cell
!mkdir -p ./data
!cp /content/drive/MyDrive/EE4146_FPL_Project/Processed_Data/ALL_SEASONS/*.csv ./data/

In [ ]:
# ============================================================
# DATA PREPARATION: Add opponent features, league positions, trends
# Produces clean final CSVs in ./final_data/
# ============================================================

import pandas as pd
import numpy as np
import os

# Paths
DATA_PATH = './data/'
OUTPUT_PATH = './final_data/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

positions = ['GK', 'DEF', 'MID', 'FWD']
raw_dfs = {}

print("="*70)
print("DATA PREPARATION FOR FPL PREDICTION")
print("="*70)

# ------------------------------------------------------------------
# 1. Load original data and standardise team names
# ------------------------------------------------------------------
print("\n📂 Step 1: Loading original CSV files...")

# Map short names → full names (to match fixture data)
team_name_map = {
    "West Ham": "West Ham United",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Wolves": "Wolverhampton Wanderers",
    "Leeds": "Leeds United",
    "Brighton": "Brighton and Hove Albion",
    "Man City": "Manchester City",
    "Man Utd": "Manchester United",
    "Spurs": "Tottenham Hotspur",
    "Leicester": "Leicester City",
}
# Add identity for names that are already full
full_names = [
    "Arsenal", "Aston Villa", "Bournemouth", "Brentford", "Chelsea",
    "Crystal Palace", "Everton", "Fulham", "Ipswich Town", "Liverpool",
    "Manchester City", "Manchester United", "Newcastle United", "Nottingham Forest",
    "Southampton", "Tottenham Hotspur", "West Ham United", "Wolverhampton Wanderers",
    "Leicester City", "Brighton and Hove Albion"
]
for name in full_names:
    team_name_map[name] = name

for pos in positions:
    file_path = os.path.join(DATA_PATH, f'{pos}_all_seasons.csv')
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing {file_path}")
    df = pd.read_csv(file_path)
    # Convert season format '23_24' → '2023-24'
    df['season'] = df['season'].str.replace('_', '-').apply(lambda x: f"20{x}" if len(x)==5 else x)
    # Normalise team names
    df['team'] = df['team'].map(team_name_map).fillna(df['team'])
    raw_dfs[pos] = df
    print(f"  ✅ Loaded {pos}: {len(df):,} rows")

all_players = pd.concat(raw_dfs.values(), ignore_index=True)
print(f"  ✅ Combined: {len(all_players):,} total rows")

# ------------------------------------------------------------------
# 2. Load fixtures and build opponent lookup (fixture ID → opponent)
# ------------------------------------------------------------------
print("\n📂 Step 2: Loading fixtures from Vaastav...")

fixture_data = []
seasons_fixture = ['2023-24', '2024-25', '2025-26']

for season in seasons_fixture:
    url = f"https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{season}/fixtures.csv"
    try:
        fixtures = pd.read_csv(url)
        fixtures['season'] = season
        fixtures.rename(columns={'event': 'GW'}, inplace=True)
        for _, row in fixtures.iterrows():
            if pd.notna(row['GW']):
                gw = int(row['GW'])
                # Home team
                fixture_data.append({
                    'season': season, 'GW': gw,
                    'team': row['team_h'], 'opponent': row['team_a'],
                    'fixture': row['id']
                })
                # Away team
                fixture_data.append({
                    'season': season, 'GW': gw,
                    'team': row['team_a'], 'opponent': row['team_h'],
                    'fixture': row['id']
                })
        print(f"  ✅ Loaded {season}")
    except Exception as e:
        print(f"  ⚠️ Could not load {season}: {e}")

fixture_df = pd.DataFrame(fixture_data)

# Map team IDs to full names
team_id_to_name = {
    1: "Arsenal", 2: "Aston Villa", 3: "Bournemouth", 4: "Brentford",
    5: "Brighton and Hove Albion", 6: "Chelsea", 7: "Crystal Palace", 8: "Everton",
    9: "Fulham", 10: "Ipswich Town", 11: "Leicester City", 12: "Liverpool",
    13: "Manchester City", 14: "Manchester United", 15: "Newcastle United",
    16: "Nottingham Forest", 17: "Southampton", 18: "Tottenham Hotspur",
    19: "West Ham United", 20: "Wolverhampton Wanderers"
}

fixture_df['team'] = fixture_df['team'].map(team_id_to_name)
fixture_df['opponent'] = fixture_df['opponent'].map(team_id_to_name)
fixture_df = fixture_df.dropna()
# Keep one row per (season, fixture, team) – avoid duplicates
fixture_df = fixture_df.drop_duplicates(subset=['season', 'fixture', 'team'])
print(f"  ✅ Created opponent lookup: {len(fixture_df):,} rows")

# ------------------------------------------------------------------
# 3. Add opponent column and clean data
# ------------------------------------------------------------------
print("\n🔗 Step 3: Adding opponent column and cleaning...")

for pos in positions:
    df = raw_dfs[pos]
    # Merge opponent using fixture ID
    df = df.merge(
        fixture_df[['season', 'fixture', 'opponent']],
        on=['season', 'fixture'],
        how='inner'               # only keep rows with valid opponent
    )
    # Remove self‑matches (team == opponent)
    df = df[df['team'] != df['opponent']].copy()
    # Deduplicate: keep first row per player per gameweek
    df = df.drop_duplicates(subset=['name', 'season', 'GW'], keep='first')
    raw_dfs[pos] = df
    print(f"  ✅ {pos}: {len(df):,} rows after cleaning")

# Rebuild all_players after cleaning
all_players = pd.concat(raw_dfs.values(), ignore_index=True)
print(f"  ✅ Combined after cleaning: {len(all_players):,} rows")

# ------------------------------------------------------------------
# 4. Calculate real league positions (from goals)
# ------------------------------------------------------------------
print("\n⚽ Step 4: Calculating real Premier League positions...")

team_goals = all_players.groupby(['season', 'GW', 'team'])['goals_scored'].sum().reset_index()

# Merge goals into fixture data
matches = fixture_df.merge(team_goals, on=['season', 'GW', 'team'], how='left')
matches.rename(columns={'goals_scored': 'team_goals'}, inplace=True)
matches = matches.merge(team_goals, left_on=['season', 'GW', 'opponent'], right_on=['season', 'GW', 'team'], how='left', suffixes=('', '_opp'))
matches.rename(columns={'goals_scored': 'opp_goals'}, inplace=True)
matches['team_goals'] = matches['team_goals'].fillna(0)
matches['opp_goals'] = matches['opp_goals'].fillna(0)

matches['match_points'] = matches.apply(lambda r: 3 if r['team_goals'] > r['opp_goals'] else (1 if r['team_goals'] == r['opp_goals'] else 0), axis=1)
matches['goal_diff'] = matches['team_goals'] - matches['opp_goals']
matches['pl_cum_points'] = matches.groupby(['season', 'team'])['match_points'].cumsum()
matches['pl_cum_gd'] = matches.groupby(['season', 'team'])['goal_diff'].cumsum()
matches['rank_metric'] = matches['pl_cum_points'] + (matches['pl_cum_gd'] / 1000.0)
matches['league_position'] = matches.groupby(['season', 'GW'])['rank_metric'].rank(ascending=False, method='min')

table_stats = matches[['season', 'GW', 'team', 'opponent', 'league_position']].drop_duplicates()
print(f"  ✅ League positions calculated for {len(table_stats):,} records")

# ------------------------------------------------------------------
# 5. Calculate opponent attack/defence points (FPL points by position)
# ------------------------------------------------------------------
print("\n📊 Step 5: Calculating opponent attack and defence points...")

position_totals = all_players.groupby(['season', 'GW', 'team', 'position'])['total_points'].sum().reset_index()

attack_points = position_totals[position_totals['position'].isin(['FWD', 'MID'])].groupby(['season', 'GW', 'team'])['total_points'].sum().reset_index()
attack_points.rename(columns={'total_points': 'opp_att_points'}, inplace=True)

defence_points = position_totals[position_totals['position'].isin(['DEF', 'GK'])].groupby(['season', 'GW', 'team'])['total_points'].sum().reset_index()
defence_points.rename(columns={'total_points': 'opp_def_points'}, inplace=True)

print(f"  ✅ Attack records: {len(attack_points):,}, Defence records: {len(defence_points):,}")

# ------------------------------------------------------------------
# 6. Merge all features into each position dataframe
# ------------------------------------------------------------------
print("\n🔗 Step 6: Merging all features...")

for pos in positions:
    df = raw_dfs[pos].copy()

    # Team's league position
    df = df.merge(table_stats[['season', 'GW', 'team', 'league_position']],
                  on=['season', 'GW', 'team'], how='left')

    # Opponent's league position
    opp_pos = table_stats[['season', 'GW', 'team', 'league_position']].rename(
        columns={'team': 'opponent', 'league_position': 'opponent_league_position'}
    )
    df = df.merge(opp_pos, on=['season', 'GW', 'opponent'], how='left')

    # Team total FPL points (sum of all players on that team)
    team_fpl = df.groupby(['season', 'GW', 'team'])['total_points'].sum().reset_index()
    team_fpl.rename(columns={'total_points': 'team_total_points'}, inplace=True)
    df = df.merge(team_fpl, on=['season', 'GW', 'team'], how='left')

    # Opponent attack points
    df = df.merge(attack_points.rename(columns={'team': 'opponent'}),
                  on=['season', 'GW', 'opponent'], how='left')

    # Opponent defence points
    df = df.merge(defence_points.rename(columns={'team': 'opponent'}),
                  on=['season', 'GW', 'opponent'], how='left')

    # Fill NaN
    df['league_position'] = df['league_position'].fillna(10)
    df['opponent_league_position'] = df['opponent_league_position'].fillna(10)
    df['team_total_points'] = df['team_total_points'].fillna(0)
    df['opp_att_points'] = df['opp_att_points'].fillna(0)
    df['opp_def_points'] = df['opp_def_points'].fillna(0)

    # Trends (3‑game rolling difference)
    df['league_pos_trend'] = df.groupby(['season', 'team'])['league_position'].transform(
        lambda x: x.diff(3).fillna(0)
    )
    df['opponent_position_trend'] = df.groupby(['season', 'opponent'])['opponent_league_position'].transform(
        lambda x: x.diff(3).fillna(0)
    )

    # Round numeric columns
    for col in ['league_position', 'opponent_league_position', 'team_total_points',
                'opp_att_points', 'opp_def_points', 'league_pos_trend', 'opponent_position_trend']:
        if col in df.columns:
            df[col] = df[col].round(2)

    # Final safety dedup
    df = df.drop_duplicates(subset=['name', 'season', 'GW'])

    # Save
    out_file = os.path.join(OUTPUT_PATH, f'{pos}_all_seasons_final.csv')
    df.to_csv(out_file, index=False)
    print(f"  ✅ Saved {pos}: {len(df):,} rows → {out_file}")

print("\n" + "="*70)
print("✅ DATA PREPARATION COMPLETE")
print(f"📁 Final CSVs saved in: {OUTPUT_PATH}")
print("="*70)

In [ ]:
# ============================================================
# FINAL VALIDATION: Random Sample for Manual Verification
# Run this AFTER data preparation
# ============================================================

import pandas as pd
import numpy as np
import os
import random

FINAL_PATH = './final_data/'
positions = ['GK', 'DEF', 'MID', 'FWD']

print("="*80)
print("MANUAL VERIFICATION SAMPLE")
print("="*80)
print("This script picks random player-gameweek entries.")
print("You can manually check the data against official sources.\n")

# Load all final CSVs
data = {}
all_players = []
for pos in positions:
    file_path = os.path.join(FINAL_PATH, f'{pos}_all_seasons_final.csv')
    if not os.path.exists(file_path):
        print(f"❌ {file_path} not found. Run data preparation first.")
        raise FileNotFoundError(f"Missing {file_path}")
    df = pd.read_csv(file_path)
    data[pos] = df
    all_players.append(df)
    print(f"✅ Loaded {pos}: {len(df):,} rows")

all_players = pd.concat(all_players, ignore_index=True)
print(f"\n📊 Combined: {len(all_players):,} rows")

# Pre‑compute team totals by position (for point breakdown)
position_totals = all_players.groupby(['season', 'GW', 'team', 'position'])['total_points'].sum().reset_index()

# Pre‑compute attack/defence lookup for quick retrieval
attack_lookup = position_totals[position_totals['position'].isin(['FWD', 'MID'])].groupby(['season', 'GW', 'team'])['total_points'].sum().reset_index()
attack_lookup.rename(columns={'total_points': 'calc_att'}, inplace=True)
defence_lookup = position_totals[position_totals['position'].isin(['DEF', 'GK'])].groupby(['season', 'GW', 'team'])['total_points'].sum().reset_index()
defence_lookup.rename(columns={'total_points': 'calc_def'}, inplace=True)

# ------------------------------------------------------------------
# Sample 10 random rows (weighted by position, but simple random is fine)
# ------------------------------------------------------------------
random.seed(42)
np.random.seed(42)

# Combine all rows from all positions into one dataframe
all_rows = pd.concat([df.assign(position=pos) for pos, df in data.items()], ignore_index=True)
sample = all_rows.sample(n=min(10, len(all_rows)), random_state=42)

print("\n" + "="*80)
print("RANDOM SAMPLE (10 entries)")
print("="*80)

for i, row in sample.iterrows():
    season = row['season']
    gw = row['GW']
    team = row['team']
    opponent = row['opponent']
    position = row['position']
    player_name = row['name']

    print(f"\n{'─'*80}")
    print(f"Sample #{i+1}: {player_name} ({position}) – {season} GW{gw}")
    print(f"{'─'*80}")

    # 1. Basic match info
    print(f"\n📋 BASIC MATCH INFO:")
    print(f"   Team: {team}")
    print(f"   Opponent: {opponent}")
    print(f"   Was Home: {row['was_home']}")
    print(f"   Clean Sheet: {row['clean_sheets']}")

    # 2. Actual match score (from goals scored)
    team_goals = all_players[(all_players['season'] == season) &
                             (all_players['GW'] == gw) &
                             (all_players['team'] == team)]['goals_scored'].sum()
    opp_goals = all_players[(all_players['season'] == season) &
                            (all_players['GW'] == gw) &
                            (all_players['team'] == opponent)]['goals_scored'].sum()
    print(f"\n⚽ ACTUAL MATCH SCORE:")
    print(f"   {team} {int(team_goals)} – {int(opp_goals)} {opponent}")
    if team_goals > opp_goals:
        print(f"   Result: WIN ✅")
    elif team_goals < opp_goals:
        print(f"   Result: LOSS ❌")
    else:
        print(f"   Result: DRAW 🤝")

    # 3. League positions (real PL table based on goals)
    print(f"\n📊 REAL PREMIER LEAGUE POSITIONS (at this GW):")
    print(f"   {team}: {row['league_position']:.0f}th")
    print(f"   {opponent}: {row['opponent_league_position']:.0f}th")
    if 'league_pos_trend' in row:
        trend = row['league_pos_trend']
        if trend > 0:
            print(f"   {team} trend: dropped {trend:.0f} places in last 3 GWs 📉")
        elif trend < 0:
            print(f"   {team} trend: climbed {abs(trend):.0f} places in last 3 GWs 📈")
        else:
            print(f"   {team} trend: stable")
    if 'opponent_position_trend' in row:
        opp_trend = row['opponent_position_trend']
        if opp_trend > 0:
            print(f"   {opponent} trend: dropped {opp_trend:.0f} places 📉")
        elif opp_trend < 0:
            print(f"   {opponent} trend: climbed {abs(opp_trend):.0f} places 📈")
        else:
            print(f"   {opponent} trend: stable")

    # 4. FPL team totals (sum of all players on that team)
    print(f"\n💰 FPL TEAM TOTALS (this GW):")
    print(f"   {team} total FPL points: {row['team_total_points']:.0f}")

    # 5. Point breakdown by position for the team
    fwd_pts = position_totals[(position_totals['season'] == season) &
                              (position_totals['GW'] == gw) &
                              (position_totals['team'] == team) &
                              (position_totals['position'] == 'FWD')]['total_points'].sum()
    mid_pts = position_totals[(position_totals['season'] == season) &
                              (position_totals['GW'] == gw) &
                              (position_totals['team'] == team) &
                              (position_totals['position'] == 'MID')]['total_points'].sum()
    def_pts = position_totals[(position_totals['season'] == season) &
                              (position_totals['GW'] == gw) &
                              (position_totals['team'] == team) &
                              (position_totals['position'] == 'DEF')]['total_points'].sum()
    gk_pts = position_totals[(position_totals['season'] == season) &
                             (position_totals['GW'] == gw) &
                             (position_totals['team'] == team) &
                             (position_totals['position'] == 'GK')]['total_points'].sum()
    print(f"\n   {team} point breakdown by position:")
    print(f"   Forwards (FWD): {fwd_pts:.0f} pts")
    print(f"   Midfielders (MID): {mid_pts:.0f} pts")
    print(f"   Defenders (DEF): {def_pts:.0f} pts")
    print(f"   Goalkeepers (GK): {gk_pts:.0f} pts")
    print(f"   TOTAL (calculated): {fwd_pts + mid_pts + def_pts + gk_pts:.0f} pts")
    print(f"   TOTAL (from team_total_points column): {row['team_total_points']:.0f} pts")

    # 6. Opponent strength features (pre‑calculated)
    print(f"\n🛡️ OPPONENT STRENGTH FEATURES (from your CSV):")
    print(f"   {opponent} Attack Points (FWD+MID): {row['opp_att_points']:.0f} pts")
    print(f"   {opponent} Defence Points (DEF+GK): {row['opp_def_points']:.0f} pts")

    # 7. Verify opponent attack/defence by direct calculation (optional check)
    calc_opp_att = attack_lookup[(attack_lookup['season'] == season) &
                                 (attack_lookup['GW'] == gw) &
                                 (attack_lookup['team'] == opponent)]['calc_att'].values
    calc_opp_def = defence_lookup[(defence_lookup['season'] == season) &
                                  (defence_lookup['GW'] == gw) &
                                  (defence_lookup['team'] == opponent)]['calc_def'].values
    calc_opp_att = calc_opp_att[0] if len(calc_opp_att) > 0 else 0
    calc_opp_def = calc_opp_def[0] if len(calc_opp_def) > 0 else 0
    print(f"\n   📝 Verification (calculated from player data):")
    print(f"   {opponent} Attack Points (FWD+MID): {calc_opp_att:.0f} pts")
    print(f"   {opponent} Defence Points (DEF+GK): {calc_opp_def:.0f} pts")
    if abs(calc_opp_att - row['opp_att_points']) < 0.5 and abs(calc_opp_def - row['opp_def_points']) < 0.5:
        print(f"   ✅ Stored values match direct calculation.")
    else:
        print(f"   ⚠️ Mismatch – stored values differ from calculation.")

    # 8. Player's own performance (for context)
    print(f"\n👤 PLAYER'S OWN STATS (this GW):")
    print(f"   Points: {row['total_points']:.0f}")
    print(f"   Goals: {row['goals_scored']:.0f}, Assists: {row['assists']:.0f}")
    print(f"   Minutes: {row['minutes']:.0f}")
    print(f"   Expected Goals (xG): {row['expected_goals']:.2f}")
    print(f"   Expected Assists (xA): {row['expected_assists']:.2f}")
    print(f"   Engineered Form (4‑game avg): {row['engineered_form']:.2f}")

    # 9. Next GW target (what we're predicting)
    print(f"\n🎯 TARGET (Next GW Points): {row['Next_GW_Points']:.0f}")

print("\n" + "="*80)
print("MANUAL VERIFICATION INSTRUCTIONS")
print("="*80)
print("""
For each sample above, you can manually verify against:
1. Premier League official website – match results and league table
2. Fantasy Premier League official website – player and team points

Check that:
- The opponent team name is correct for that GW.
- The match score (goals) matches real life.
- The league positions match the actual PL table after that GW.
- The FPL team totals and point breakdowns are plausible.
- The opponent attack/defence points make sense (sum of their players' points).
- The player's own stats (goals, assists, minutes) match reality.

If everything looks correct, your data is ready for training.
""")